# Hướng dẫn Code AWS Lambda (ETL bằng awswrangler)
Tài liệu này chứa các đoạn code mẫu và **những điểm quan trọng cần chú ý** khi bạn tự triển khai Lambda để xử lý các file CSV sang Parquet, đúng như thỏa thuận là chỉ đề xuất mà không can thiệp sửa code dự án của bạn.

## 1. Import và Cấu hình cơ bản
**🚨 CHÚ Ý QUAN TRỌNG:**
- Bạn **phải đóng gói `awswrangler` (AWS SDK for pandas) thành Lambda Layer**. Không cài trực tiếp bằng `pip install` vào chung thư mục code Lambda vì dung lượng của thư viện này (và các phụ thuộc như pandas/numpy) sẽ vượt quá giới hạn 250MB của Lambda.
- Khi chạy với **LocalStack** thay vì AWS thật, bạn phải trỏ endpoint của AWS services về `localhost:4566`.

In [ ]:
import urllib.parse
import awswrangler as wr
import pandas as pd
import boto3

# CHÚ Ý (LocalStack): awswrangler cần cấu hình boto3 client hướng về endpoint của LocalStack.
# Trong môi trường AWS thật, bạn BỎ QUA phần khai báo session này.
session = boto3.Session()
s3_client = session.client('s3', endpoint_url='http://localhost:4566')

## 2. Hàm xử lý chính (Lambda Handler)
**🚨 CHÚ Ý QUAN TRỌNG:**
- Luôn trích xuất đúng tên bucket và object key từ biến `event`.
- Nhớ sử dụng hàm `urllib.parse.unquote_plus` vì S3 event key thường bị mã hóa URL (ví dụ dấu cách trong tên file bị biến thành dấu `+`). Nếu không unquote, Lambda sẽ báo lỗi không tìm thấy file.
- Không nên đặt file đích và file gốc vào cùng 1 bucket để tránh hiện tượng vòng lặp vô hạn (Infinite Loop) tự kích hoạt Lambda.

In [ ]:
def lambda_handler(event, context):
    # Lấy thông tin bucket và file từ S3 trigger event
    bucket = event['Records'][0]['s3']['bucket']['name']
    key = urllib.parse.unquote_plus(event['Records'][0]['s3']['object']['key'], encoding='utf-8')
    
    source_path = f"s3://{bucket}/{key}"
    dest_bucket = "process-bucket" # Tên bucket đích đã khai báo trong init_s3.sh
    
    # Tạo đường dẫn đích: Thay đổi đuôi file từ .csv sang .parquet
    # Ví dụ: data/interactions.csv -> s3://process-bucket/data/interactions.parquet
    dest_key = key.replace('.csv', '.parquet')
    dest_path = f"s3://{dest_bucket}/{dest_key}"
    
    print(f"Bắt đầu xử lý: {source_path} -> {dest_path}")
    
    # 1. Đọc và Transform dữ liệu
    df = process_file(source_path)
    
    # 2. Ghi dữ liệu ra định dạng Parquet
    write_parquet(df, dest_path)
    
    return {
        'statusCode': 200,
        'body': f'Đã xử lý thành công {key}'
    }

## 3. Hàm Đọc và Chuẩn hóa dữ liệu (Stateless Transform)
**🚨 CHÚ Ý QUAN TRỌNG:**
- Để hàm Lambda giữ trạng thái **Stateless (độc lập)**, chúng ta không thực hiện join các file ở đây. 
- Nếu file cực kì lớn, bạn có thể truyền tham số `chunksize=10000` vào hàm `read_csv`, nhưng với `awswrangler` cơ bản đã tối ưu C++ bên dưới, việc load các file dưới 500MB khá thoải mái với mức RAM Lambda 1024MB.

In [ ]:
def process_file(s3_path):
    # Dùng awswrangler đọc trực tiếp CSV từ S3. 
    # Truyền tham số boto3_session chỉ khi dùng LocalStack.
    df = wr.s3.read_csv(path=s3_path, boto3_session=session) 
    
    # --- CÁC BƯỚC DATA CLEANSING KHUYẾN NGHỊ ---
    
    # 1. Chuẩn hóa tên cột (viết thường, khoảng trắng -> gạch dưới) để tiện truy vấn bằng Athena sau này
    df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]
    
    # 2. Xử lý ép kiểu ngày tháng tự động cho các cột chứa từ 'time' hoặc 'date'
    # (Ví dụ: cột event_time, created_at,...)
    for col in df.columns:
        if 'time' in col or 'date' in col:
            try:
                df[col] = pd.to_datetime(df[col], errors='ignore')
            except Exception:
                pass
                
    # 3. Điền giá trị rỗng (Tùy chọn)
    # Ví dụ: df.fillna({'price': 0.0}, inplace=True)
    
    return df

## 4. Hàm Ghi dữ liệu định dạng Parquet
**🚨 CHÚ Ý QUAN TRỌNG:**
- Parquet là định dạng lưu trữ theo cột (columnar) giúp tiết kiệm dung lượng (nhẹ hơn CSV tới 50-80%) và giữ nguyên kiểu dữ liệu (Data Type).
- Bắt buộc sử dụng `index=False`, nếu không Pandas sẽ tạo thêm 1 cột index dư thừa vào file Parquet làm tốn dung lượng.

In [ ]:
def write_parquet(df, s3_dest_path):
    # Ghi thẳng DataFrame sang định dạng Parquet trên S3
    wr.s3.to_parquet(
        df=df,
        path=s3_dest_path,
        boto3_session=session, # Xóa tham số này trên AWS thật
        index=False # Luôn để False
    )
    print(f"Đã lưu thành công định dạng Parquet tại {s3_dest_path}")